In [1]:
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                              f1_score, precision_score, recall_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [4]:
import os
TARGET_ROWS = 30000
TOLERANCE = 0.15

notebook_dir = os.getcwd()


data_path = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed", "pr_snapshots_clean_v2.csv"))

# 3. Read the file
df = pd.read_csv(data_path)

repo_sizes = df.groupby("repo_key").size().sample(frac=1, random_state=42)
keep_repos, total = [], 0
for repo, size in repo_sizes.items():
    if total >= TARGET_ROWS:
        break
    if total + size > TARGET_ROWS * (1 + TOLERANCE):
        continue
    keep_repos.append(repo)
    total += size

df = df[df.repo_key.isin(keep_repos)].copy()
print(f"{len(keep_repos)} repos, {len(df)} rows")

y = df.pop("merged_before_next")
X = df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")

5 repos, 30867 rows


In [5]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["repo_key"]))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
train_repo_keys = df["repo_key"].iloc[train_idx]  # for GroupKFold below -- repo-level, not pr_id
print(f"train repos: {train_repo_keys.nunique()}, test repos: {df.repo_key.iloc[test_idx].nunique()}")

train repos: 4, test repos: 1


In [7]:
# -1 + missing-indicator for sklearn-native models; XGBoost/LightGBM get raw NaN (native handling)
imputer = SimpleImputer(strategy="constant", fill_value=-1, add_indicator=True)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=imputer.get_feature_names_out(), index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=imputer.get_feature_names_out(), index=X_test.index)

sw_train = compute_sample_weight(class_weight="balanced", y=y_train)

In [8]:
models_config = {
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]},
        True,
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=42),
        {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15, None], "min_samples_split": [2, 5, 10]},
        True,
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=42),
        {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]},
        True,
    ),
    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=42),
        {"max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]},
        False,
    ),
    "LightGBM": (
        LGBMClassifier(random_state=42, verbose=-1),
        {"max_depth": [3, 5, 7, -1], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300], "subsample": [0.7, 0.8, 1.0]},
        False,
    ),
}

gkf = GroupKFold(n_splits=3)
results = []

for name, (model, params, needs_imputation) in models_config.items():
    start_time = time.time()
    X_tr = X_train_imp if needs_imputation else X_train
    X_te = X_test_imp if needs_imputation else X_test

    search = RandomizedSearchCV(model, params, n_iter=10, scoring="balanced_accuracy",
                                 cv=gkf, random_state=42, n_jobs=-1)
    search.fit(X_tr, y_train, groups=train_repo_keys, sample_weight=sw_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_te)
    y_prob = best_model.predict_proba(X_te)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob),
        "F1 Score": f1_score(y_test, y_pred, average="weighted"),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "Time Taken (s)": time.time() - start_time,
        "Best Params": str(search.best_params_),
    })

benchmark_df = pd.DataFrame(results).sort_values("Balanced Accuracy", ascending=False).set_index("Model")
benchmark_df.round(4)

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
Model,,,,,,,,
Random Forest,0.6270,0.6616,0.6857,0.6335,0.7003,0.6270,43.4650,"{'n_estimators': 200, 'min_samples_split': 5, ..."
LightGBM,0.6250,0.6541,0.6815,0.6322,0.6905,0.6250,12.6396,"{'subsample': 0.7, 'n_estimators': 200, 'max_d..."
AdaBoost,0.6200,0.6479,0.6588,0.6275,0.6843,0.6200,17.4906,"{'n_estimators': 50, 'learning_rate': 0.5}"
Extra Trees,0.6324,0.6475,0.6893,0.6406,0.6791,0.6324,25.9136,"{'n_estimators': 300, 'min_samples_split': 10,..."
XGBoost,0.6237,0.6462,0.6850,0.6317,0.6805,0.6237,5.8938,"{'subsample': 0.8, 'n_estimators': 100, 'max_d..."


In [9]:

# 1. Load Data
notebook_dir = os.getcwd()
data_path = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed", "pr_snapshots_clean_v2.csv"))
df = pd.read_csv(data_path)

# Create a working copy to preserve the original 'df'
working_df = df.copy()

# 2. Prepare Features and Target
y = working_df.pop("merged_before_next")
X = working_df.select_dtypes(include="number").drop(columns=["pr_id", "number", "timeline_data_available"], errors="ignore")
print(f"shape: {X.shape}, target distribution: {y.value_counts().to_dict()}")

# 3. Group Split: Hold out exactly 2 whole repos for testing
splitter = GroupShuffleSplit(n_splits=1, test_size=8, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=working_df["repo_key"]))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print(f"train repos: {working_df.repo_key.iloc[train_idx].nunique()}, test repos: {working_df.repo_key.iloc[test_idx].nunique()}")

# 4. Extract PR IDs specifically for the inner GroupKFold tuning
train_pr_ids = working_df["repo_key"].iloc[train_idx]

# 5. Impute Data (with -1 constant and indicator flag)
imputer = SimpleImputer(strategy="constant", fill_value=-1, add_indicator=True)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=imputer.get_feature_names_out(), index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=imputer.get_feature_names_out(), index=X_test.index)

# 6. Compute Sample Weights for balanced training
sw_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 7. Configure Random Forest and RandomizedSearchCV
model = RandomForestClassifier(random_state=42)
params = {
    "n_estimators": [100, 200, 300], 
    "max_depth": [5, 10, 15, None], 
    "min_samples_split": [2, 5, 10]
}

gkf = GroupKFold(n_splits=3)

search = RandomizedSearchCV(
    estimator=model, 
    param_distributions=params, 
    n_iter=10, 
    scoring="balanced_accuracy", 
    cv=gkf, 
    random_state=42, 
    n_jobs=-1
)

# 8. Train and Tune
print("Training Random Forest...")
start_time = time.time()

# Pass groups to prevent PR leakage, and sample_weight to handle imbalance
search.fit(X_train_imp, y_train, groups=train_pr_ids, sample_weight=sw_train)
best_model = search.best_estimator_

# 9. Evaluate
y_pred = best_model.predict(X_test_imp)
y_prob = best_model.predict_proba(X_test_imp)
roc_auc = roc_auc_score(y_test, y_prob[:, 1]) if len(set(y_test)) == 2 else roc_auc_score(y_test, y_prob, multi_class='ovr')

# 10. Display Results
results = pd.DataFrame([{
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "ROC AUC": roc_auc,
    "F1 Score": f1_score(y_test, y_pred, average='weighted'),
    "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
    "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
    "Time Taken (s)": time.time() - start_time,
    "Best Params": str(search.best_params_)
}])

display(results.round(4))

shape: (364705, 24), target distribution: {0: 259682, 1: 105023}
train repos: 33, test repos: 8
Training Random Forest...


,Model,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken (s),Best Params
0,Random Forest,0.7094,0.6915,0.7663,0.7192,0.7399,0.7094,634.6583,"{'n_estimators': 300, 'min_samples_split': 10,..."


In [11]:
import joblib

save_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "output", "savedmodels"))

os.makedirs(save_dir, exist_ok=True)

file_path = os.path.join(save_dir, "rf_pr_bottleneck_model.pkl")

artifacts = {
    "model": best_model,
    "imputer": imputer
}

# Save it to disk
joblib.dump(artifacts, file_path)
print(f"\nModel and imputer successfully saved to:\n{file_path}")


Model and imputer successfully saved to:
C:\Users\VEDANG BARMAN\Desktop\Git_Pr_prediction\output\savedmodels\rf_pr_bottleneck_model.pkl
